# 01 — Data layer

**Phase 1 deliverable:** three cached Parquet files and a printed quality report for each.

The notebooks in this project are *outputs*, not sources. Every line of logic lives in
`src/stock_retrofit/`; these cells only call it. That keeps the package testable and keeps
the notebooks readable.

> **yfinance is the primary source in this build.** Not by preference — the Settrade Open API is
> credential-gated behind a broker relationship and no credentials exist in this environment.
> Spec R2 explicitly provides for this fallback. See `docs/settrade-api-notes.md` for what was
> tried and what remains open.

In [8]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
import pandas as pd
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
print("stock-retrofit @", ROOT)

stock-retrofit @ /Users/kbstudio/Library/CloudStorage/OneDrive-Personal/Project/model/stock-retrofit


## Fetch and cache

The only network path in the package. Everything downstream reads the Parquet cache (spec R3).
Each write lands a `.meta.json` sidecar recording source, timestamp, row count, date range and a
content hash, so any result traces back to the exact bytes that produced it.

In [9]:
from datetime import date
from stock_retrofit.data import fetch

metas = fetch(["KBANK", "SCB", "BAY"], source="yfinance", start=date(2000, 1, 1))
for symbol, meta in metas.items():
    print(f"{symbol:6s} {meta.rows:5d} bars  {meta.start} .. {meta.end}  "
          f"hash {meta.content_hash[:12]}  repairs {meta.repairs['count']}")

KBANK   6594 bars  2000-01-04 .. 2026-08-13  hash 7f890f001da6  repairs 3
SCB     1047 bars  2022-04-20 .. 2026-08-13  hash 009611b3e1c1  repairs 1
BAY     6594 bars  2000-01-04 .. 2026-08-13  hash 4ccb5ee8ae79  repairs 6


## Quality gates

Two tiers, deliberately. **Structural violations raise** — `high < low`, a close outside
`[low, high]`, duplicate dates, or an *unexplained* move beyond SET's ±30% band. None of those are
possible in correct data. **Advisories are reported** — missing sessions, zero-volume days. Those
are real facts about a thin market, not corruption.

In [10]:
from stock_retrofit.data import quality_report

for symbol in ["KBANK", "SCB", "BAY"]:
    print(quality_report(symbol).render())
    print()

Data quality — KBANK: 6594 bars, 2000-01-04 .. 2026-08-13
  sessions expected................. 6594
  mean daily volume................. 9,432,486
  median daily volume............... 6,915,550
  daily return sd................... 2.0223%
  largest 1-day move................ 18.44%
  registered breaks................. none
  structural violations................ none
  advisories:
    - 193 zero-volume sessions (2.93% of bars) — no tradeable liquidity on those days
    - 3 vendor bar defect(s) repaired under policy 'widen_bar_to_close' on 3 date(s): 2021-03-01, 2023-09-12, 2023-10-11 — see the .meta.json sidecar for the full audit trail

Data quality — SCB: 1047 bars, 2022-04-20 .. 2026-08-13
  sessions expected................. 1051
  mean daily volume................. 11,355,130
  median daily volume............... 9,117,500
  daily return sd................... 1.7637%
  largest 1-day move................ 39.88%
  registered breaks................. 1
  structural violations..........

### What the gate found, and why each is handled the way it is

**Vendor bar defects.** 3 bars in KBANK, 1 in SCB, 6 in BAY have a `close` outside `[low, high]`
by one to three ticks. Repair is a *separate stage* from the gate: the gate still raises on
anything handed to it unrepaired, and every repair is recorded in the meta sidecar. A bad bar
being fixed on the record is fine; a bad bar passing silently is the thing this layer exists to
prevent.

**SCB's 4-session gap and +39.9% move.** Trading halted 2022-04-21 → 2022-04-26 around the issuer
substitution, and the price discontinuity lands on 2022-04-27, the first session back. The gate
treats a move that *spans* a registered break as explained rather than as a limit violation —
matching on the break date alone would have missed it.

In [11]:
import json
from stock_retrofit.paths import RAW_DIR

print(json.dumps(json.loads((RAW_DIR / "BAY.meta.json").read_text())["repairs"], indent=2)[:1200])

{
  "policy": "widen_bar_to_close",
  "count": 6,
  "records": [
    {
      "date": "2023-05-23",
      "field": "high",
      "old": 29.0,
      "new": 29.25,
      "reason": "open/close exceeded the reported high"
    },
    {
      "date": "2023-06-16",
      "field": "high",
      "old": 32.0,
      "new": 32.5,
      "reason": "open/close exceeded the reported high"
    },
    {
      "date": "2023-07-24",
      "field": "low",
      "old": 31.25,
      "new": 31.0,
      "reason": "open/close fell below the reported low"
    },
    {
      "date": "2023-08-09",
      "field": "high",
      "old": 32.25,
      "new": 32.5,
      "reason": "open/close exceeded the reported high"
    },
    {
      "date": "2023-08-15",
      "field": "low",
      "old": 31.75,
      "new": 31.5,
      "reason": "open/close fell below the reported low"
    },
    {
      "date": "2023-10-05",
      "field": "high",
      "old": 30.75,
      "new": 31.0,
      "reason": "open/close exceeded the repo

## Instrument semantics

The SCB caveat lives in code, not in someone's memory (spec R13–R15). A caller can never receive a
series spanning a registered break without a signal that it did.

In [12]:
from stock_retrofit.data import describe

for symbol in ["KBANK", "SCB", "BAY"]:
    print(describe(symbol)); print()

KBANK — Kasikornbank PCL
  liquidity: Large cap, liquid, no known discontinuity. The clean case.

SCB — SCB X PCL (formerly The Siam Commercial Bank PCL)
  liquidity: Large cap, liquid. Series carries an issuer substitution at 2022-04-22.
  break 2022-04-22 [issuer_substitution]: SCB delisted and SCB X PCL listed 1:1 in its place, retaining the SCB ticker. A change of issuer (bank -> holding company), not merely of name. Pre-2022-04-22 'SCB' bars belong to a different legal entity.
    source: SCB/SCBX first-party announcements, March-April 2022

BAY — Bank of Ayudhya PCL (Krungsri)
  liquidity: Thin float: ~72-76% held by MUFG since the 2013 acquisition. Daily turnover is small relative to SCB/KBANK. Treat as the liquidity stress case and cap participation in any backtest.
  default participation cap: 5.0% of volume



In [13]:
from stock_retrofit.data import load

truncated = load("SCB")                                   # default policy
flagged   = load("SCB", policy="full_with_changepoint")
print("truncate_at_break     :", len(truncated), "bars from", truncated["date"].min().date())
print("full_with_changepoint :", len(flagged), "bars, changepoints flagged:",
      int(flagged["is_changepoint"].sum()))

truncate_at_break     : 1046 bars from 2022-04-27
full_with_changepoint : 1047 bars, changepoints flagged: 0


**An empirical finding worth recording:** Yahoo's `SCB.BK` series *begins* 2022-04-20 — the
vendor already carries SCBX only. So `truncate_at_break` is satisfied trivially and
`full_with_changepoint` cannot be populated from this source; there is no pre-break history to
keep. The policy flag and registry entry are implemented and tested regardless, because the
caveat belongs in code and because a future Settrade fetch could supply the missing years.

## Reconciliation

Spec R10 wants two independent sources cross-checked, never averaged. With only one source
reachable this degrades to a cache-vs-vendor check — which catches staleness and vendor revisions
but *cannot* catch an error Yahoo makes consistently. The output labels the actual sources so the
limitation is visible rather than implied.

In [14]:
from stock_retrofit.data import reconcile

table = reconcile("KBANK", against="yfinance")
print(f"{len(table)} overlapping dates, "
      f"{int(table['exceeds_one_tick'].sum())} exceed one tick of disagreement")
table.tail()

6594 overlapping dates, 1 exceed one tick of disagreement


,date,close_yfinance,close_yfinance,diff,abs_diff,tick,exceeds_one_tick
6589,2026-08-06,252.0,252.0,0.0,0.0,1.0,False
6590,2026-08-07,250.0,250.0,0.0,0.0,1.0,False
6591,2026-08-10,248.0,248.0,0.0,0.0,1.0,False
6592,2026-08-11,249.0,249.0,0.0,0.0,1.0,False
6593,2026-08-13,247.0,249.0,-2.0,2.0,1.0,True
